# Supplementary Figure 14 | Per-trait GWAS–QTL CoS counts by QTL biotype

**a.** For each GWAS trait, bars show the number of cross-tissue ColocBoost GWAS–QTL CoS (GWAS p < 10⁻⁵). Bars are colored by the biotypes of colocalized QTLs. Traits are labeled with abbreviations from Barbeira et al. 2021 and with the total number of GWAS–QTL CoS for that trait.

**b.** Same as (a) but for genome-wide significant loci (p < 5.7 × 10⁻⁹).


## Setup

In [ ]:
from collections import Counter
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=5)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("font", size=6)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

np.random.seed(35934)

lncrna_color = "#B71C1C"
p_gws = 5.7e-9
fig_width, fig_height, fig_dpi = 7.2, 5.4, 300
bar_h = 0.72


def as_str_list(val):
    """Normalize parquet list/array/scalar cells to a list of non-empty strings."""
    if val is None:
        return []
    if isinstance(val, (list, tuple, np.ndarray, pd.Series)):
        return [
            str(v)
            for v in list(val)
            if v is not None and str(v) not in ("", "nan", "None", "<NA>")
        ]
    try:
        if pd.isna(val):
            return []
    except (TypeError, ValueError):
        pass
    s = str(val).strip()
    return [s] if s and s not in ("nan", "None", "<NA>") else []


_ensg_stem = re.compile(r"(ENSG\d+)")


def as_stem_set(vals):
    """Extract stable ENSG IDs from versioned / composite QTL gene IDs."""
    out = set()
    if vals is None:
        return out
    try:
        if isinstance(vals, float) and np.isnan(vals):
            return out
    except (TypeError, ValueError):
        pass
    items = vals if isinstance(vals, (list, tuple, np.ndarray, set, pd.Series)) else [vals]
    for v in items:
        if v is None:
            continue
        m = _ensg_stem.search(str(v))
        if m:
            out.add(m.group(1))
    return out


fine_incl_order = [
    "novel_only", "lnc_only", "novel_shared", "lnc_shared", "other",
]
_reds = sns.color_palette("Reds", 8)
fine_incl_colors = {
    "novel_only": lncrna_color,
    "lnc_only": _reds[5],
    "novel_shared": _reds[3],
    "lnc_shared": _reds[2],
    "other": "#9E9E9E",
}
fine_incl_labels = {
    "novel_only": "novel lncRNA only",
    "lnc_only": "lncRNA only",
    "novel_shared": "novel lncRNA + other",
    "lnc_shared": "lncRNA + other",
    "other": "other gene",
}

# Zenodo ColocBoost helpers
def tissue_from_cos_id(cid):
    s = str(cid)
    i = s.rfind("_")
    return s[:i] if i > 0 else s


def lead_from_lists(variants, vcp):
    vs = list(variants)
    vcps = list(vcp)
    ps = pd.to_numeric(pd.Series(vcps), errors="coerce")
    if len(ps) == len(vs) and ps.notna().any():
        return vs[int(ps.values.argmax())]
    return vs[0]


def cos_to_long(row):
    vs = list(row["variants"])
    ps = list(row["vcp"])
    return pd.DataFrame({
        "variant_id": vs,
        "vcp": ps[: len(vs)] + [float("nan")] * (len(vs) - len(ps)),
    })


def _coloc_list_len(val):
    if val is None:
        return 0
    if isinstance(val, float) and np.isnan(val):
        return 0
    if isinstance(val, np.ndarray):
        return len(val)
    if isinstance(val, (list, tuple, set)):
        return len(val)
    return 0


def _typeset(vals):
    if vals is None:
        return set()
    if isinstance(vals, float) and np.isnan(vals):
        return set()
    if isinstance(vals, np.ndarray):
        items = vals.tolist()
    elif isinstance(vals, (list, tuple, set)):
        items = list(vals)
    else:
        items = [vals]
    out = set()
    for v in items:
        if v is None:
            continue
        s = str(v).strip()
        if s and s not in ("nan", "None", "<NA>"):
            out.add(s)
    return out


def _member_qtl_class(types):
    t = _typeset(types)
    lnc = t & {"novel_lncRNA", "existing_lncRNA"}
    non = t - {"novel_lncRNA", "existing_lncRNA"}
    if not lnc:
        return "non-lncRNA"
    if non:
        return "Shared"
    return "lncRNA-specific (robust)"


def _locus_qtl_label(member_classes):
    labs = set(member_classes)
    if labs == {"lncRNA-specific (robust)"}:
        return "lncRNA-specific (robust)"
    if "Shared" in labs or ("non-lncRNA" in labs and "lncRNA-specific (robust)" in labs):
        return "Shared"
    return "non-lncRNA"


def load_main_colocboost(colocboost_dir, *, cols=None):
    frames = []
    for path in sorted(colocboost_dir.glob("*.ColocBoost.parquet")):
        if path.name.startswith("cross_"):
            continue
        tissue = path.name.replace(".ColocBoost.parquet", "")
        df = pd.read_parquet(path, columns=cols)
        df["tissue"] = tissue
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def build_cross_tissue_members(cross_tissue_loci_path, colocboost_dir, *, include_ucos=False):
    loci = pd.read_parquet(cross_tissue_loci_path)
    all_cos = load_main_colocboost(colocboost_dir).set_index("cos_id", drop=False)
    memb = loci.explode("member_cos_ids").rename(columns={"member_cos_ids": "cos_id"})
    memb["cos_id"] = memb["cos_id"].astype(str)
    cos_rows = memb[["locus_id", "cos_id"]].merge(
        all_cos.reset_index(drop=True), on="cos_id", how="inner"
    )
    cos_rows["lead_variant_id"] = [
        lead_from_lists(v, p) for v, p in zip(cos_rows["variants"], cos_rows["vcp"])
    ]
    if "phenotype_id" not in cos_rows.columns:
        cos_rows["phenotype_id"] = cos_rows["phenotype_ids"]

    qtl_by_locus = (
        cos_rows.groupby("locus_id")["gene_types_robust"]
        .agg(lambda s: _locus_qtl_label(_member_qtl_class(x) for x in s))
    )
    locus_summary = loci.copy()
    locus_summary["qtl_label"] = locus_summary["locus_id"].map(qtl_by_locus)

    if not include_ucos:
        return locus_summary, cos_rows

    member_ids = set(memb["cos_id"])
    has_gwas = all_cos["gwas_traits"].map(_coloc_list_len) > 0
    has_qtl = all_cos["gene_ids_robust"].map(_coloc_list_len) > 0
    ucos = all_cos.loc[
        ~all_cos.index.isin(member_ids) & (has_gwas ^ has_qtl)
    ].reset_index(drop=True)
    ucos = ucos.loc[has_gwas.reindex(ucos["cos_id"]).fillna(False).values].copy()
    ucos["locus_id"] = "ucos_" + ucos["cos_id"].astype(str)
    ucos["lead_variant_id"] = [
        lead_from_lists(v, p) for v, p in zip(ucos["variants"], ucos["vcp"])
    ]
    if "phenotype_id" not in ucos.columns:
        ucos["phenotype_id"] = ucos["phenotype_ids"]
    members = pd.concat([cos_rows, ucos], ignore_index=True)
    return locus_summary, members



## Paths

In [ ]:
gencode_class = Path("path/to/gencode_v39_v47_gene_classification.txt")
trait_abbr_tsv = Path("path/to/table_s2_abbreviations.txt")
gwas_meta_file = Path("path/to/barbeira_full_gwas_metadata.txt")

colocboost_dir = Path("path/to")
loci_path = Path("path/to/cross_tissue_loci.parquet")

fig_dir = Path("figures")
fig_pdf = fig_dir / "supp_figure_14_coloc_gwas.pdf"
fig_png = fig_dir / "supp_figure_14_coloc_gwas.png"


## Load data

### GENCODE gene classification

In [ ]:
_gc = pd.read_csv(gencode_class, sep="\t", low_memory=False)
lncrna_ids = set(
    _gc.loc[
        _gc["gene_type_grouped"].isin(["novel_lncRNA", "existing_lncRNA"]),
        "gene_id_stable",
    ].astype(str)
)
novel_lncrna_ids = set(
    _gc.loc[
        _gc["gene_type_grouped"] == "novel_lncRNA",
        "gene_id_stable",
    ].astype(str)
)
print(f"lncRNA stems: {len(lncrna_ids):,}; novel: {len(novel_lncrna_ids):,}")

### Trait abbreviations (Table S2)

In [ ]:
s2_abbr = set()
with open(trait_abbr_tsv) as fh:
    fh.readline()
    for line in fh:
        parts = line.strip().split()
        if parts:
            s2_abbr.add(parts[-2])

### GWAS metadata (trait tag → abbreviation)

In [ ]:
gwas_meta = pd.read_csv(gwas_meta_file, sep="\t")
trait_labels = {}
for tag, code in zip(gwas_meta["Tag"], gwas_meta["new_abbreviation"]):
    if pd.isna(code) or code not in s2_abbr:
        continue
    trait_labels[str(tag)] = str(code)
print(f"Table S2 trait labels: {len(trait_labels)}")

### Cross-tissue ColocBoost CoS members

In [ ]:
locus_summary, locus_members = build_cross_tissue_members(loci_path, colocboost_dir, include_ucos=False
)
locus_members["member_has_qtl"] = locus_members["gene_ids_robust"].map(
    lambda v: _coloc_list_len(v) > 0
)
locus_members["gene_stems"] = locus_members["gene_ids_robust"].map(as_stem_set)
# Main cross-tissue loci are GWAS–QTL CoS only (no uCoS table / recovery denom).
locus_members = locus_members.loc[locus_members["member_has_qtl"]].copy()
print(
    f"ColocBoost members (GWAS–QTL CoS): {len(locus_members):,} rows; "
    f"loci={locus_members['locus_id'].nunique():,}"
)


## Run analysis

In [ ]:
def lnc_inclusion_class_fine(gene_stems) -> str:
    """5-way QTL gene inclusion (mutually exclusive).

    novel_only: only novel lncRNA
    lnc_only: only lncRNA, but not exclusively novel (existing and/or mixed)
    novel_shared: novel lncRNA + non-lncRNA (may also include existing lncRNA)
    lnc_shared: existing lncRNA + non-lncRNA (no novel)
    other: no lncRNA
    """
    if not gene_stems:
        return "other"
    has_novel = any(g in novel_lncrna_ids for g in gene_stems)
    has_lnc = any(g in lncrna_ids for g in gene_stems)
    has_other = any(g not in lncrna_ids for g in gene_stems)
    if has_lnc and not has_other:
        if has_novel and all(g in novel_lncrna_ids for g in gene_stems):
            return "novel_only"
        return "lnc_only"
    if has_novel and has_other:
        return "novel_shared"
    if has_lnc and has_other:
        return "lnc_shared"
    return "other"


# CoS-max -log10(p) cuts (same as combine / Barbeira).
_GWS_NEGLOG10 = float(-np.log10(5.7e-9))
_SUB_NEGLOG10 = float(-np.log10(1e-5))


def _parse_neglog10_list(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []
    out = []
    for tok in str(val).split(","):
        tok = tok.strip()
        if not tok or tok.lower() == "nan":
            out.append(np.nan)
            continue
        try:
            out.append(float(tok))
        except ValueError:
            out.append(np.nan)
    return out


def gwas_traits_above_cut(phenotype_id, pval_nominal_phenotypes, cut) -> list:
    """GWAS trait names with CoS-max -log10(p) strictly above ``cut``."""
    parts = [p.strip() for p in str(phenotype_id or "").split(";") if p.strip()]
    vals = _parse_neglog10_list(pval_nominal_phenotypes)
    out = []
    for i, part in enumerate(parts):
        if i >= len(vals):
            break
        pl = part.lower()
        if "expression" in pl or "splicing" in pl:
            continue
        v = vals[i]
        if np.isfinite(v) and float(v) > cut:
            out.append(part)
    return out


def traits_at_1e5(row) -> list:
    return sorted(set(gwas_traits_above_cut(
        getattr(row, "phenotype_ids", None),
        getattr(row, "cos_max_pval_nominal_phenotypes", None),
        _SUB_NEGLOG10,
    )))


def traits_at_gws(row) -> list:
    return sorted(set(as_str_list(getattr(row, "gwas_gws_traits", None))))


def build_fine_trait_counts(gwas_df, trait_fn):
    """Collapse members → counts per trait of GWAS–QTL CoS; 5-way gene inclusion."""
    trait_to_loci = {}
    for r in gwas_df.itertuples(index=False):
        lid = str(getattr(r, "locus_id"))
        stems = getattr(r, "gene_stems", set()) or set()
        for trait in trait_fn(r):
            if trait not in trait_labels:
                continue
            slot = trait_to_loci.setdefault(trait, {})
            if lid not in slot:
                slot[lid] = set()
            slot[lid] |= set(stems)

    rows = []
    for trait, loci in trait_to_loci.items():
        stack_fine = Counter(
            lnc_inclusion_class_fine(genes) for genes in loci.values()
        )
        n_joint = sum(stack_fine.values())
        fine_row = {
            "trait": trait,
            "abbr": trait_labels[trait],
            "n_joint": n_joint,
        }
        for key in fine_incl_order:
            fine_row[f"n_{key}"] = int(stack_fine.get(key, 0))
        rows.append(fine_row)
    return (
        pd.DataFrame(rows)
        .sort_values(["n_joint", "abbr"], ascending=[False, True])
        .reset_index(drop=True)
    )


# Panel a: CoS-max p < 1e-5
_has_1e5 = [
    bool(gwas_traits_above_cut(pid, pval, _SUB_NEGLOG10))
    for pid, pval in zip(
        locus_members["phenotype_ids"], locus_members["cos_max_pval_nominal_phenotypes"]
    )
]
gwas_1e5 = locus_members.loc[_has_1e5].copy()
n_loci_1e5 = int(gwas_1e5["locus_id"].nunique())
trait_recovery_fine_df = build_fine_trait_counts(gwas_1e5, traits_at_1e5)

# Panel b: GWS p < 5.7e-9
_has_gws = locus_members["gwas_gws_traits"].map(lambda v: len(as_str_list(v)) > 0)
gwas_gws = locus_members.loc[_has_gws].copy()
n_loci_gws = int(gwas_gws["locus_id"].nunique())
trait_recovery_fine_gws_df = build_fine_trait_counts(gwas_gws, traits_at_gws)

print(
    f"a unique GWAS–QTL CoS (p<1e-5): {n_loci_1e5:,}; "
    f"trait×locus={int(trait_recovery_fine_df['n_joint'].sum()):,}"
)
print(
    f"b unique GWAS–QTL CoS (p<{p_gws:.1e}): {n_loci_gws:,}; "
    f"trait×locus={int(trait_recovery_fine_gws_df['n_joint'].sum()):,}"
)
print(f"traits plotted: a={len(trait_recovery_fine_df)}, b={len(trait_recovery_fine_gws_df)}")


## Make figure

In [ ]:
def _overall_counts(df):
    return {k: int(df[f"n_{k}"].sum()) for k in fine_incl_order}


def _stack_bar(ax, y, counts, height=bar_h, lw=0.25):
    left = 0.0
    for key in fine_incl_order:
        w = float(counts[key])
        if w <= 0:
            continue
        ax.barh(
            y, w, left=left, height=height,
            color=fine_incl_colors[key], edgecolor="black",
            linewidth=lw, zorder=2,
        )
        left += w


def _plot_fine_panel(ax, df, xlabel):
    # Absolute counts: omit a pooled Overall bar (sum would dwarf every trait).
    df = df.sort_values(
        ["n_joint", "abbr"], ascending=[False, True]
    ).reset_index(drop=True)
    n_traits = len(df)
    ys = np.arange(n_traits)[::-1]
    xmax = 0.0
    for y, row in zip(ys, df.itertuples(index=False)):
        counts = {k: getattr(row, f"n_{k}") for k in fine_incl_order}
        _stack_bar(ax, y, counts)
        xmax = max(xmax, float(sum(counts.values())))
    yticklabels = [
        f"{abbr} (n={int(n):,})"
        for abbr, n in zip(df["abbr"], df["n_joint"])
    ]
    ax.set_yticks(list(ys))
    ax.set_yticklabels(yticklabels, fontsize=3.2)
    ax.set_ylim(-1, n_traits)
    ax.set_xlim(0, xmax * 1.05 if xmax else 1)
    ax.set_xlabel(xlabel)
    ax.tick_params(axis="y", length=0, pad=1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return _overall_counts(df), df


fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(fig_width, fig_height), dpi=fig_dpi)
overall_a, trait_recovery_fine_df = _plot_fine_panel(
    ax_a,
    trait_recovery_fine_df.copy(),
    r"Cross-tissue GWAS–QTL CoS ($p < 1 \times 10^{-5}$)",
)
overall_b, trait_recovery_fine_gws_df = _plot_fine_panel(
    ax_b,
    trait_recovery_fine_gws_df.copy(),
    r"Cross-tissue GWAS–QTL CoS ($p < 5.7 \times 10^{-9}$)",
)

fig.subplots_adjust(top=0.86, wspace=0.42)
legend_handles = [
    Patch(
        facecolor=fine_incl_colors[k], edgecolor="black", linewidth=0.4,
        label=fine_incl_labels[k],
    )
    for k in fine_incl_order
]
fig.legend(
    handles=legend_handles,
    loc="lower center", bbox_to_anchor=(0.5, 0.865),
    ncol=5, frameon=False, fontsize=4.5,
    handletextpad=0.3, columnspacing=0.7, borderaxespad=0.0,
)
pos_a = ax_a.get_position()
pos_b = ax_b.get_position()
fig.text(pos_a.x0, 0.905, "a", fontsize=9, fontweight="bold", va="bottom", ha="left")
fig.text(pos_b.x0, 0.905, "b", fontsize=9, fontweight="bold", va="bottom", ha="left")
plt.show()

## Save figure

In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_pdf, bbox_inches="tight")
fig.savefig(fig_png, bbox_inches="tight")
print(f"Wrote {fig_pdf}")
print(f"Wrote {fig_png}")

## Print stats

In [ ]:
# No numeric values from this figure are cited in the paper text or caption.
def _print_overall(label, counts):
    n = sum(counts.values())
    parts = ", ".join(
        f"{fine_incl_labels[k]}={counts[k]:,} ({100.0 * counts[k] / n:.1f}%)"
        for k in fine_incl_order
        if counts[k]
    )
    print(f"{label} pooled trait×locus n={n:,}: {parts}")


_print_overall("a", overall_a)
_print_overall("b", overall_b)
